# B1 — Introduction to Dynamic Linear Models

**Reference:** West & Harrison §1–2; Petris §1

**Concepts introduced:**
- The state space model: observation and state equations
- Why we need a filter: the latent state is unobserved
- Predict–update cycle: the Kalman filter algorithm

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from engine.filter import kalman_filter
from engine.models import make_local_level
from engine.simulate import simulate

## 1. The two-equation model

A **Dynamic Linear Model** describes how an unobserved state $\theta_t$ evolves
and how noisy observations $y_t$ are generated from it (W&H §2.2):

$$\boxed{y_t = F_t \theta_t + v_t, \quad v_t \sim N(0, V_t)}  \quad \text{(observation equation)}$$

$$\boxed{\theta_t = G_t \theta_{t-1} + w_t, \quad w_t \sim N(0, W_t)}  \quad \text{(state equation)}$$

**What each piece means:**
- $\theta_t$ — the true underlying quantity we care about (latent, unobserved)
- $y_t$ — what we actually measure (noisy version of $F_t \theta_t$)
- $F_t$ — how the state maps to the observation (often just 1)
- $G_t$ — how the state evolves from one step to the next
- $V_t$ — observation noise: how noisy is each measurement?
- $W_t$ — state noise: how much does the true state wander?

**The filtering problem:** given $y_1, \ldots, y_t$, what is the distribution
$p(\theta_t \mid y_{1:t})$? The Kalman filter answers this exactly when the model
is Gaussian.

## 2. The simplest DLM: local level

The **local level model** (W&H §4.3) sets $F=1$, $G=1$:

$$y_t = \theta_t + v_t, \quad v_t \sim N(0, V)$$
$$\theta_t = \theta_{t-1} + w_t, \quad w_t \sim N(0, W)$$

$\theta_t$ is a random-walk level. $y_t$ is that level observed with noise $V$.
The ratio $\kappa = W/V$ (signal-to-noise) controls how quickly the filter tracks changes.

In [ ]:
# Simulate a local-level series by hand — no engine yet
rng = np.random.default_rng(0)
T = 80
V_true, W_true = 2.0, 0.5

theta = np.zeros(T)
y     = np.zeros(T)

theta[0] = rng.normal(0.0, 1.0)
y[0]     = theta[0] + rng.normal(0.0, np.sqrt(V_true))

for t in range(1, T):
    theta[t] = theta[t-1] + rng.normal(0.0, np.sqrt(W_true))   # state evolves
    y[t]     = theta[t]   + rng.normal(0.0, np.sqrt(V_true))   # noisy observation

t_arr = np.arange(T)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_arr, theta, "k-", lw=1.5, label="true state θ_t")
ax.plot(t_arr, y, "C0.", ms=4, alpha=0.6, label="observations y_t")
ax.set(xlabel="t", ylabel="value", title=f"Local level simulation  (V={V_true}, W={W_true})")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Obs std (expected √V={np.sqrt(V_true):.2f}): {y.std():.2f}")

## 3. The Kalman filter: predict–update

The filter maintains a Gaussian belief $p(\theta_t \mid y_{1:t}) = N(m_t, C_t)$.
Each time step has two phases (W&H §2.3):

### Predict (prior for time $t$)

$$a_t = G\, m_{t-1} \qquad (\text{prior mean})$$
$$R_t = G\, C_{t-1}\, G' + W \qquad (\text{prior covariance})$$

### Update (incorporate $y_t$)

$$e_t = y_t - F\, a_t \qquad (\text{innovation: surprise in observation})$$
$$Q_t = F\, R_t\, F' + V \qquad (\text{innovation covariance})$$
$$A_t = R_t\, F'\, Q_t^{-1} \qquad (\text{Kalman gain: how much to trust } y_t)$$
$$m_t = a_t + A_t\, e_t \qquad (\text{posterior mean})$$
$$C_t = R_t - A_t\, Q_t\, A_t' \qquad (\text{posterior covariance})$$

**Intuition:** if $Q_t$ is large (uncertain prediction), $A_t$ is large — we update
strongly toward $y_t$. If $Q_t$ is small (confident prediction), $A_t$ is small — we
barely move.

In [ ]:
# Manual filter step at t=1
# Prior at t=0: m0=0, C0=1000 (diffuse)
F, G = np.array([[1.0]]), np.array([[1.0]])
V_mat = np.array([[V_true]])
W_mat = np.array([[W_true]])

# --- Predict ---
m_prev = np.array([0.0])
C_prev = np.array([[1000.0]])
a1 = G @ m_prev                            # (1,)
R1 = G @ C_prev @ G.T + W_mat             # (1,1)

# --- Update ---
e1   = y[0] - (F @ a1)[0]                 # scalar innovation
Q1   = F @ R1 @ F.T + V_mat               # (1,1)
A1   = R1 @ F.T @ np.linalg.inv(Q1)       # (1,1) Kalman gain
m1   = a1 + (A1 @ np.array([[e1]]))[: , 0]   # (1,) posterior mean
C1   = R1 - A1 @ Q1 @ A1.T               # (1,1) posterior covariance

print(f"y[0]      = {y[0]:.4f}")
print(f"a1        = {a1[0]:.4f}  (prior mean: no data yet → 0)")
print(f"e1        = {e1:.4f}  (innovation)")
print(f"A1        = {A1[0,0]:.6f}  (Kalman gain)")
print(f"m1        = {m1[0]:.4f}  (posterior mean)")
print(f"√C1       = {np.sqrt(C1[0,0]):.4f}  (posterior std)")

In [ ]:
# Full filter via engine
spec = make_local_level(V=V_true, W_level=W_true)
fr   = kalman_filter(spec, y[:, None])   # y must be (T,1)

fig, ax = plt.subplots(figsize=(11, 4))
std = np.sqrt(fr.C[:, 0, 0])
ax.plot(t_arr, theta, "k-", lw=1, label="true θ_t")
ax.plot(t_arr, y, ".", ms=4, alpha=0.5, color="C0", label="obs y_t")
ax.plot(t_arr, fr.m[:, 0], "C1-", lw=2, label="filtered mean m_t")
ax.fill_between(t_arr,
                fr.m[:, 0] - 1.96*std,
                fr.m[:, 0] + 1.96*std,
                alpha=0.25, color="C1", label="95% filter interval")
ax.set(xlabel="t", ylabel="value", title="Kalman filter on local level")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Log marginal likelihood: {fr.loglik:.2f}")

## Exercises

**Exercise 1** — Change W from 0.5 to 5.0 and re-run the engine filter. What happens
to the 95% interval? Does the filtered mean track $y_t$ more or less closely? Why?

**Exercise 2** — Trace the predict step manually for $t=2$. Use `m1` and `C1` computed
above as the prior, then compute `a2`, `R2` with the formulas:
```python
a2 = G @ m1
R2 = G @ C1 @ G.T + W_mat
```
Compare `a2` to `fr.a[1, 0]` — they should match (remember `fr` is 0-indexed, so t=2 maps to index 1).